<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_02_optimiser_comparison.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 02 — Four Optimisers, One Problem

**Paired with L6.1 · Loss Functions and Gradients**

Notebook 01 asked where the loss comes from. This one asks how you get to the
bottom of it, and it is the notebook that earns its keep in Part 2: every PINN
in L7 to L12 is trained by **Adam first, then L-BFGS**, and this is where you
find out why that handoff exists rather than being told.

## What you will do

1. Fit the damped structural response with the same network four times,
   changing only the optimiser.
2. Sweep the learning rate and watch SGD fail in two different directions.
3. Meet L-BFGS, whose interface is different because it evaluates the loss
   more than once per step.
4. Chain Adam into L-BFGS and measure what the handoff buys.

## One thing to notice before you start

The target is **noiseless** — `core.response_dataset` adds nothing. That is
deliberate, and notebook 00 said so. With noise, every optimiser stops at the
same floor and the comparison measures the noise instead of the optimiser.
Here the floor is zero, so the differences between these four are visible for
as long as you care to train.

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_6_core as core

core.set_seed(0)
print("torch", torch.__version__, "| output dir:", core.OUTPUT_DIR)

## 1 · The problem

Two hundred noiseless samples of `exp(-0.9x) sin(4x)`, and a network with two
hidden layers of sixteen units. Small enough to train in seconds, wiggly enough
that a badly tuned optimiser visibly fails.

In [ ]:
x, y = core.response_dataset(n=200)
X, Y = core.to_tensor(x), core.to_tensor(y)
loss_fn = nn.MSELoss()

core.set_seed(0)
print("parameters in the model:", core.count_parameters(core.MLP(hidden=(16, 16))))

fig, ax = plt.subplots(figsize=(7.0, 3.6))
ax.plot(x, y, lw=1.9, color="#1f77b4")
ax.set_xlabel("$x$"); ax.set_ylabel("$y$")
ax.set_title("The target: a damped structural response, sampled without noise")
ax.grid(alpha=0.25)
plt.show()

### Your turn

One function, used everywhere below. Keeping the seed reset **inside** it is
what makes the four runs comparable: every optimiser starts from an identical
set of weights, so any difference you see is the optimiser and not the draw.

In [ ]:
# TODO: write run(make_opt, epochs) -> (history, model).
#
#   def run(make_opt, epochs=2000):
#       core.set_seed(0)                     # identical initial weights, always
#       model = core.MLP(hidden=(16, 16))
#       opt = make_opt(model.parameters())
#       history = []
#       for _ in range(epochs):
#           opt.zero_grad()
#           loss = loss_fn(model(X), Y)
#           loss.backward()
#           opt.step()
#           history.append(loss.item())
#       return np.array(history), model
#
# make_opt is a function of the parameters, not an optimiser object -- an
# optimiser is bound to the parameters of one model, so it cannot be reused
# across runs.

raise NotImplementedError("Write run(make_opt, epochs)")

## 2 · Three optimisers at the same budget

Same initial weights, same two thousand epochs, same data. The only thing that
changes is the update rule.

In [ ]:
FIRST_ORDER = {
    "SGD":            lambda p: torch.optim.SGD(p, lr=0.05),
    "SGD + momentum": lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9),
    "Adam":           lambda p: torch.optim.Adam(p, lr=0.01),
}

histories, models = {}, {}
for name, make in FIRST_ORDER.items():
    histories[name], models[name] = run(make, epochs=2000)
    print(f"  {name:<16s} final loss {histories[name][-1]:.6f}")

core.plot_curves(histories, title="Same network, same budget, three update rules")
plt.show()

**What you should see.** Three curves separated by orders of magnitude, plain
SGD highest. Momentum and Adam both get further, and Adam gets there sooner.

The gap is not that SGD is a bad algorithm. It is that plain SGD applies the
same step size to every parameter, and this loss surface is far steeper in some
directions than others — so the step that is right for the steep directions is
far too small for the flat ones.

## 3 · The learning rate decides more than the optimiser does

Before concluding anything about update rules, check whether you have merely
compared three learning rates. Sweep it.

### Your turn

In [ ]:
# TODO: sweep the learning rate for SGD and for Adam.
#
#   LRS = [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3]
#
#   For each lr, run BOTH
#       lambda p, lr=lr: torch.optim.SGD(p, lr=lr)
#       lambda p, lr=lr: torch.optim.Adam(p, lr=lr)
#   for 800 epochs, and record the FINAL loss.
#
#   Beware the closure bug: `lambda p: torch.optim.SGD(p, lr=lr)` inside a loop
#   captures the variable lr, not its value, so every entry ends up using the
#   last one. The `lr=lr` default argument above is what pins it.
#
#   A run can diverge to inf or nan. Record it as np.nan rather than letting it
#   through -- that IS the result, and section 4 plots it.
#
#   Put them in sweep = {"SGD": [...], "Adam": [...]}, aligned with LRS.

raise NotImplementedError("Sweep the learning rate for SGD and Adam")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
for i, (name, losses) in enumerate(sweep.items()):
    ax.plot(LRS, losses, "o-", lw=1.9, ms=6, color=["#d94f2b", "#1f77b4"][i],
            label=name)
    for lr, v in zip(LRS, losses):
        if not np.isfinite(v):
            ax.plot([lr], [ax.get_ylim()[1]], "x", ms=11, mew=2.4,
                    color=["#d94f2b", "#1f77b4"][i])
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("learning rate"); ax.set_ylabel("loss after 800 epochs")
ax.set_title("A cross marks a run that diverged")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

print(core.error_table(
    [[f"{lr:g}"] + [("diverged" if not np.isfinite(sweep[n][i])
                     else f"{sweep[n][i]:.5f}") for n in sweep]
     for i, lr in enumerate(LRS)],
    ["learning rate"] + list(sweep)))

**What you should see.** Both curves are U-shaped, and Adam's U is *wider*.
That width is the honest summary of what adaptive methods buy you: not a better
optimum, but a much larger range of learning rates that reach a decent one.

Read the two failure directions off the plot, because they look nothing alike
in a training log:

* **Too small** — the loss falls smoothly and stops early. Nothing looks wrong.
  This is the dangerous one, because the run *looks* converged.
* **Too large** — the loss oscillates, then leaves for infinity or `nan`. Ugly,
  obvious, and much easier to diagnose.

## 4 · L-BFGS, and why its interface is different

Everything above takes one gradient and one step. L-BFGS builds an
approximation to the curvature from the last few gradients and uses it to
choose both a direction and a distance — then runs a line search along that
direction, which means **evaluating the loss several times per step**.

So it cannot be handed a single backward pass. It needs a function it can call
repeatedly. That function is the *closure*, and it must do the full
zero-grad / forward / backward each time it is called.

Two consequences worth remembering when you meet this again in L7:

* L-BFGS is a **full-batch** method. It assumes the loss it is shown is the
  same function each time. Mini-batches break that assumption.
* It is fast near a minimum and unreliable far from one, which is exactly the
  opposite of Adam.

In [ ]:
core.set_seed(0)
lbfgs_model = core.MLP(hidden=(16, 16))
opt = torch.optim.LBFGS(lbfgs_model.parameters(), lr=1.0, max_iter=20)

def closure():
    opt.zero_grad()
    loss = loss_fn(lbfgs_model(X), Y)
    loss.backward()
    return loss

lbfgs_history = []
for _ in range(60):
    opt.step(closure)
    with torch.no_grad():
        lbfgs_history.append(loss_fn(lbfgs_model(X), Y).item())

print(f"L-BFGS from a cold start, 60 steps: {lbfgs_history[-1]:.8f}")

## 5 · The handoff Part 2 relies on

Adam is robust far from a solution and slow to polish. L-BFGS is the reverse.
The standard recipe in the PINN literature — and in every exercise from Ex_07
onwards — is to use each where it is strong: **Adam to get close, L-BFGS to
finish**.

### Your turn

In [ ]:
# TODO: chain the two, and compare against equal-budget alternatives.
#
#   Build FOUR results, all starting from core.set_seed(0) and core.MLP((16,16)):
#
#     "Adam 2000"            Adam(lr=0.01), 2000 epochs
#     "L-BFGS 60"            L-BFGS as in section 4, 60 steps, from cold
#     "Adam 1000 -> L-BFGS"  Adam(lr=0.01) for 1000 epochs, then L-BFGS on the
#                            SAME model for 60 steps
#     "Adam 2000 -> L-BFGS"  Adam for 2000 epochs, then L-BFGS for 60 steps
#
#   Record the final loss of each in final = {name: loss}.
#
#   For the chained runs, build the L-BFGS optimiser AFTER Adam has finished,
#   over the same model.parameters(). A closure closes over the model, so
#   define it after the model exists.

raise NotImplementedError("Chain Adam into L-BFGS")

In [ ]:
print(core.error_table(
    [[name, f"{loss:.3e}"] for name, loss in final.items()],
    ["recipe", "final training loss"]))

**What you should see.** The chained runs reach a loss several orders of
magnitude below Adam alone, and cold L-BFGS lands somewhere unimpressive — it
is a local method handed a bad starting point.

This is the entire justification for the two-stage training you will type
without thinking for the next six weeks. It is worth having measured it once.

**A caution that matters in Part 2.** A loss of `1e-8` on a PINN residual is
not evidence that the solution is right. It says the network satisfies the
equations you wrote at the points you sampled. If the boundary condition is
wrong, or the collocation points miss a feature, L-BFGS will drive that wrong
problem to machine precision very efficiently. Convergence is not correctness.

## 6 · What this notebook does not show

Say these out loud so the comparison is not oversold:

* **No stochasticity.** Full-batch throughout, so "SGD" here is gradient
  descent. Mini-batch noise changes the picture and sometimes helps.
* **No generalisation.** Training loss only, on noiseless data. The optimiser
  that fits the training set best is not automatically the one you want —
  Ex_04 notebook 05 is the counterexample.
* **One problem, one architecture, one seed for the initial weights.** The
  ordering here is typical, not universal.

## 7 · Save

In [ ]:
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb02_optimisers.npz")
np.savez(path,
         lrs=np.asarray(LRS),
         sweep_sgd=np.asarray(sweep["SGD"], dtype=float),
         sweep_adam=np.asarray(sweep["Adam"], dtype=float),
         hist_sgd=histories["SGD"],
         hist_momentum=histories["SGD + momentum"],
         hist_adam=histories["Adam"],
         hist_lbfgs=np.asarray(lbfgs_history),
         final_names=np.array(list(final)),
         final_losses=np.asarray(list(final.values()), dtype=float))
print("wrote", path)

## 8 · Before you move on

You should be able to answer these without rerunning anything:

1. Why is the target noiseless, and what would noise have hidden?
2. Which failure mode of a badly chosen learning rate is harder to spot in a
   training log, and why?
3. Why does L-BFGS need a closure when Adam does not?
4. Why is cold L-BFGS worse than Adam-then-L-BFGS, given that L-BFGS uses more
   information per step?

Next: **notebook 03**, where a trained network meets a second machine and most
of what it learned turns out to still be useful.